# FireSight-IR | Module 3a — LinkedIn Visualization

Generates publication-quality figures from Module 3a training log and evaluation results.

**Run after Module 3a evaluation cell (Section 9 of 03a notebook).**  
Variables `val_probs`, `val_labels`, `test_probs`, `test_labels`, `log` must be in memory,  
OR run Section 1 below which reloads everything from Drive.

## Figures produced
1. **LinkedIn hero** — 4-panel composite: training curves, confusion matrix, ROC curves, BTD physics verification
2. **Training curves detail** — standalone loss + per-class accuracy
3. **Probability distributions** — wildfire probability density by true class

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install torch matplotlib scikit-learn numpy pandas h5py scipy tqdm -q

import os, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.ticker import FuncFormatter
from scipy import stats
from sklearn.metrics import (
    confusion_matrix, roc_curve, roc_auc_score,
    classification_report
)
import numpy._core.multiarray
torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_GPU = device.type == 'cuda'
print(f'Device: {device}')
print('Imports OK.')

---
## Section 1 — Load everything from Drive

In [ ]:
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'
MODEL_DIR   = f'{BASE_DIR}/models'
FIGURES_DIR = f'{BASE_DIR}/figures'
CACHE_DIR   = f'{BASE_DIR}/data/cache'
SPLIT_DIR   = f'{BASE_DIR}/data/splits'
BEST_PATH   = f'{MODEL_DIR}/firesight_pinn_best.pt'
LOG_PATH    = f'{MODEL_DIR}/training_log.json'
os.makedirs(FIGURES_DIR, exist_ok=True)

N_ATM=16; N_SRF=20; N_DERIVED=6; N_CLASSES=3; DROPOUT=0.3

# ── Re-declare model ──────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self,i,o,d=0.2):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(i,o),nn.BatchNorm1d(o),nn.ReLU(),nn.Dropout(d),nn.Linear(o,o),nn.BatchNorm1d(o))
        self.proj=nn.Linear(i,o) if i!=o else nn.Identity()
    def forward(self,x): return F.relu(self.net(x)+self.proj(x))
class CNNBranch(nn.Module):
    def __init__(self,c=4,d=0.2):
        super().__init__()
        self.enc=nn.Sequential(
            nn.Conv2d(c,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(True),nn.Conv2d(32,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(True),nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(True),nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(True),nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(True),nn.MaxPool2d(2))
        self.gap=nn.AdaptiveAvgPool2d(1); self.drop=nn.Dropout(d)
    def forward(self,x): return self.drop(self.gap(self.enc(x)).flatten(1))
class FireSightPINN(nn.Module):
    def __init__(self,na=16,ns=20,nd=6,nc=3,dr=0.3):
        super().__init__()
        self.cnn=CNNBranch(4,dr)
        self.atm=nn.Sequential(ResidualBlock(na,64),ResidualBlock(64,32))
        self.srf=nn.Sequential(ResidualBlock(ns,64),ResidualBlock(64,32))
        self.der=nn.Sequential(nn.Linear(nd,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),nn.Linear(32,16),nn.BatchNorm1d(16),nn.ReLU())
        self.fusion=nn.Sequential(nn.Linear(208,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(dr),nn.Linear(128,64),nn.BatchNorm1d(64),nn.ReLU(),nn.Dropout(dr))
        self.cls=nn.Linear(64,nc)
        self.trans=nn.Sequential(nn.Linear(64,16),nn.ReLU(),nn.Linear(16,1),nn.Sigmoid())
    def forward(self,p,a,s,d):
        f=self.fusion(torch.cat([self.cnn(p),self.atm(a),self.srf(s),self.der(d)],dim=1))
        return self.cls(f),self.trans(f)

# Load model
model = FireSightPINN(N_ATM,N_SRF,N_DERIVED,N_CLASSES,DROPOUT).to(device)
ckpt  = torch.load(BEST_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✓ Model loaded: epoch {ckpt["epoch"]} | val_loss={ckpt["val_loss"]:.4f}')

# Load training log
with open(LOG_PATH) as f:
    raw_log = json.load(f)

# Deduplicate: keep lowest val_loss per epoch
best_per_epoch = {}
for e in raw_log:
    ep = e['epoch']
    if ep not in best_per_epoch or e['val_loss'] < best_per_epoch[ep]['val_loss']:
        best_per_epoch[ep] = e
log = [best_per_epoch[ep] for ep in sorted(best_per_epoch)]
print(f'Training log: {len(log)} unique epochs (best val_loss per epoch)')

In [ ]:
# ── Dataset and evaluation ────────────────────────────────────────────────────
CACHE_FILES = {k: f'{CACHE_DIR}/{k}.npy'
               for k in ['patches','atm','srf','derived','labels','aux']}

class FireDS(Dataset):
    def __init__(self,cf,idx):
        self.idx=np.sort(idx)
        self.p=np.load(cf['patches'],mmap_mode='r'); self.a=np.load(cf['atm'],mmap_mode='r')
        self.s=np.load(cf['srf'],mmap_mode='r');     self.d=np.load(cf['derived'],mmap_mode='r')
        self.l=np.load(cf['labels'],mmap_mode='r');  self.x=np.load(cf['aux'],mmap_mode='r')
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        n=int(self.idx[i])
        return (torch.from_numpy(self.p[n].copy()),torch.from_numpy(self.a[n].copy()),
                torch.from_numpy(self.s[n].copy()),torch.from_numpy(self.d[n].copy()),
                torch.tensor(int(self.l[n]),dtype=torch.long),
                torch.from_numpy(self.x[n].copy()))

val_idx  = np.load(f'{SPLIT_DIR}/val_index.npy')
test_idx = np.load(f'{SPLIT_DIR}/test_index.npy')

@torch.no_grad()
def evaluate(indices, desc=''):
    loader = DataLoader(FireDS(CACHE_FILES,indices), batch_size=2048, shuffle=False, num_workers=0)
    preds,lbls,probs,aux_list=[],[],[],[]
    for p,a,s,d,l,x in loader:
        with autocast(enabled=HAS_GPU):
            logits,trans=model(p.to(device),a.to(device),s.to(device),d.to(device))
        preds.append(logits.argmax(1).cpu().numpy())
        lbls.append(l.numpy())
        probs.append(F.softmax(logits,dim=1).cpu().numpy())
        aux_list.append(x.numpy())
    return (np.concatenate(preds), np.concatenate(lbls),
            np.concatenate(probs), np.concatenate(aux_list))

print('Running evaluation...')
val_preds, val_labels, val_probs, val_aux   = evaluate(val_idx,  'Val')
test_preds,test_labels,test_probs,test_aux  = evaluate(test_idx, 'Test')

# BT_I4 and BTD from aux cache (cols: [BT_I4_centre, BTD_centre, beer_lambert_proxy])
val_bt4  = val_aux[:,0];  val_btd  = val_aux[:,1]
test_bt4 = test_aux[:,0]; test_btd = test_aux[:,1]

print(f'Val : {len(val_labels):,} pixels | acc={(val_preds==val_labels).mean():.4f}')
print(f'Test: {len(test_labels):,} pixels | acc={(test_preds==test_labels).mean():.4f}')
print('✓ Evaluation complete.')

---
## Section 2 — LinkedIn hero figure (4-panel composite)

In [ ]:
# ── Design system ─────────────────────────────────────────────────────────────
BG      = '#080E14'
PANEL   = '#0F1923'
PANEL2  = '#141E2B'
BORDER  = '#1E2D3D'
TEXT    = '#E8F4FD'
SUBTEXT = '#6B8FA8'
ORANGE  = '#FF6B1A'
ORANGE2 = '#FF8C42'
NAVY    = '#2563EB'
TEAL    = '#14B8A6'
GREEN   = '#22C55E'
RED     = '#EF4444'
GOLD    = '#F59E0B'
PURPLE  = '#A855F7'

CLS_COLORS = [TEAL, ORANGE, NAVY]   # non-fire, wildfire, false-alarm
CLS_NAMES  = ['Non-fire', 'Wildfire', 'False-alarm']

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': PANEL,
    'axes.edgecolor': BORDER, 'text.color': TEXT,
    'xtick.color': SUBTEXT, 'ytick.color': SUBTEXT,
    'axes.labelcolor': SUBTEXT, 'grid.color': BORDER,
    'grid.linewidth': 0.5, 'lines.linewidth': 2,
})

# Extract log arrays (deduplicated)
eps    = [e['epoch']      for e in log]
trl    = [e['train_loss'] for e in log]
vll    = [e['val_loss']   for e in log]
tra    = [e['train_acc']  for e in log]
vla    = [e['val_acc']    for e in log]
wf_acc = [e['per_class_val_acc'].get('wf',0) for e in log]
fa_acc = [e['per_class_val_acc'].get('fa',0) for e in log]
nf_acc = [e['per_class_val_acc'].get('nf',0) for e in log]
ce_l   = [e['loss_components']['ce'] for e in log]
bl_l   = [e['loss_components']['bl'] for e in log]

best_ep_idx = int(np.argmin(vll))
best_ep     = eps[best_ep_idx]

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 12), facecolor=BG)
fig.patch.set_facecolor(BG)

gs = gridspec.GridSpec(2, 3, figure=fig,
                       left=0.06, right=0.97,
                       top=0.87, bottom=0.07,
                       wspace=0.35, hspace=0.50)

# ── Panel A: Training + validation loss ───────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
ax_a.set_facecolor(PANEL)
ax_a.plot(eps, trl, color=SUBTEXT,  lw=1.5, linestyle='--', alpha=0.7, label='Train loss')
ax_a.plot(eps, vll, color=ORANGE,   lw=2.5, label='Val loss (2023)')
ax_a.axvline(best_ep, color=GOLD, lw=1.5, linestyle=':', alpha=0.8)
ax_a.text(best_ep+0.3, max(vll)*0.88, f'Best\nepoch {best_ep}\n{min(vll):.4f}',
          fontsize=7.5, color=GOLD, va='top')
ax_a.set_title('Loss convergence', color=TEXT, fontsize=11, pad=8)
ax_a.set_xlabel('Epoch', fontsize=9)
ax_a.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax_a.spines[['top','right']].set_visible(False)
ax_a.grid(alpha=0.25)
# Annotate physics loss
ax_a2 = ax_a.twinx()
ax_a2.plot(eps, bl_l, color=TEAL, lw=1.2, linestyle=':', alpha=0.6, label='Beer-Lambert')
ax_a2.set_ylabel('Beer-Lambert loss', fontsize=7, color=TEAL)
ax_a2.tick_params(axis='y', colors=TEAL, labelsize=7)
ax_a2.spines[['top','left']].set_visible(False)
ax_a2.set_facecolor(PANEL)

# ── Panel B: Per-class validation accuracy ────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
ax_b.set_facecolor(PANEL)
ax_b.plot(eps, [v*100 for v in wf_acc], color=ORANGE, lw=2.5, label='Wildfire')
ax_b.plot(eps, [v*100 for v in fa_acc], color=NAVY,   lw=2.5, label='False-alarm')
ax_b.plot(eps, [v*100 for v in nf_acc], color=TEAL,   lw=2,   linestyle='--', alpha=0.8, label='Non-fire')
ax_b.axhline(95, color=GOLD, lw=1, linestyle=':', alpha=0.6)
ax_b.text(1, 95.5, '95% target', fontsize=7.5, color=GOLD)
ax_b.axvline(best_ep, color=GOLD, lw=1.5, linestyle=':', alpha=0.6)
ax_b.set_ylim(70, 102)
ax_b.set_title('Per-class accuracy (val 2023)', color=TEXT, fontsize=11, pad=8)
ax_b.set_xlabel('Epoch', fontsize=9)
ax_b.set_ylabel('Recall (%)', fontsize=9)
ax_b.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax_b.spines[['top','right']].set_visible(False)
ax_b.grid(alpha=0.25)

# ── Panel C: ROC curves ────────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[0, 2])
ax_c.set_facecolor(PANEL)
ax_c.plot([0,1],[0,1], color=BORDER, lw=1, linestyle='--', alpha=0.6)
for ci, (name, color) in enumerate(zip(CLS_NAMES, CLS_COLORS)):
    bl = (val_labels == ci).astype(int)
    if bl.sum() == 0: continue
    fpr, tpr, _ = roc_curve(bl, val_probs[:, ci])
    auc = roc_auc_score(bl, val_probs[:, ci])
    lw  = 3 if ci == 2 else 2
    ax_c.plot(fpr, tpr, color=color, lw=lw,
              label=f'{name}  AUC={auc:.4f}')
ax_c.set_xlabel('False positive rate', fontsize=9)
ax_c.set_ylabel('True positive rate', fontsize=9)
ax_c.set_title('ROC curves (val 2023)', color=TEXT, fontsize=11, pad=8)
ax_c.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax_c.spines[['top','right']].set_visible(False)
ax_c.grid(alpha=0.25)
ax_c.set_xlim(-0.01, 1.01)
ax_c.set_ylim(-0.01, 1.02)
# Annotate perfect AUC
ax_c.annotate('AUC = 1.0000\nPerfect separation',
              xy=(0.08, 0.80), xytext=(0.3, 0.55),
              fontsize=7.5, color=NAVY,
              arrowprops=dict(arrowstyle='->', color=NAVY, lw=1.2))

# ── Panel D: Wildfire probability distribution by true class ──────────────────
ax_d = fig.add_subplot(gs[1, 0])
ax_d.set_facecolor(PANEL)
for ci, (name, color) in enumerate(zip(CLS_NAMES, CLS_COLORS)):
    sub = val_probs[val_labels==ci, 1]  # wildfire probability
    if len(sub) < 10: continue
    xr = np.linspace(0, 1, 400)
    kde = stats.gaussian_kde(sub, bw_method=0.08)
    d   = kde(xr); d = d/d.max()
    ax_d.fill_between(xr, d, alpha=0.25, color=color)
    ax_d.plot(xr, d, color=color, lw=2.5,
              label=f'{name}  (n={len(sub):,})')
ax_d.axvline(0.5, color=GOLD, lw=1.2, linestyle=':', alpha=0.7, label='Decision boundary')
ax_d.set_xlabel('P(wildfire) — Model output', fontsize=9)
ax_d.set_ylabel('Normalised density', fontsize=9)
ax_d.set_title('Wildfire probability separation', color=TEXT, fontsize=11, pad=8)
ax_d.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax_d.spines[['top','right']].set_visible(False)
ax_d.grid(alpha=0.25)

# ── Panel E: BTD physics verification ─────────────────────────────────────────
ax_e = fig.add_subplot(gs[1, 1])
ax_e.set_facecolor(PANEL)
# Sample for speed
rng = np.random.default_rng(42)
for ci, (name, color) in enumerate(zip(CLS_NAMES, CLS_COLORS)):
    mask = val_labels == ci
    if mask.sum() < 10: continue
    n_s  = min(8000, mask.sum())
    idx  = rng.choice(np.where(mask)[0], n_s, replace=False)
    btd_s= val_btd[idx]
    xr   = np.linspace(max(btd_s.min(),-5), min(btd_s.max(),90), 300)
    kde  = stats.gaussian_kde(btd_s, bw_method=0.25)
    d    = kde(xr); d = d/d.max()
    ax_e.fill_between(xr, d, alpha=0.22, color=color)
    ax_e.plot(xr, d, color=color, lw=2.5,
              label=f'{name}  med={np.median(btd_s):.1f}K')
ax_e.axvline(10, color=GOLD, lw=1.8, linestyle=':', alpha=0.8,
             label='BTD=10K\nphysics threshold')
ax_e.set_xlabel('BTD = BT_I4 − BT_I5  (K)', fontsize=9)
ax_e.set_ylabel('Normalised density', fontsize=9)
ax_e.set_title('Brightness temp. difference\n(physics constraint validation)', color=TEXT, fontsize=11, pad=8)
ax_e.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax_e.spines[['top','right']].set_visible(False)
ax_e.grid(alpha=0.25)

# ── Panel F: Confusion matrix (val 2023) ──────────────────────────────────────
ax_f = fig.add_subplot(gs[1, 2])
ax_f.set_facecolor(PANEL)
cm = confusion_matrix(val_labels, val_preds, normalize='true')
im = ax_f.imshow(cm, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax_f.set_xticks(range(3)); ax_f.set_yticks(range(3))
ax_f.set_xticklabels(['Non-fire','Wildfire','False-alarm'], rotation=25, ha='right', fontsize=9, color=TEXT)
ax_f.set_yticklabels(['Non-fire','Wildfire','False-alarm'], fontsize=9, color=TEXT)
ax_f.set_xlabel('Predicted', fontsize=9, color=SUBTEXT)
ax_f.set_ylabel('True', fontsize=9, color=SUBTEXT)
for i in range(3):
    for j in range(3):
        v = cm[i,j]
        color_txt = 'white' if v < 0.55 else '#080E14'
        fw = 'bold' if i == j else 'normal'
        ax_f.text(j, i, f'{v:.3f}', ha='center', va='center',
                  color=color_txt, fontsize=10, fontweight=fw)
plt.colorbar(im, ax=ax_f, fraction=0.04, pad=0.04)
ax_f.set_title('Confusion matrix (val 2023)', color=TEXT, fontsize=11, pad=8)

# ── Header ────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.97,
         'FireSight-IR  |  Module 3a — Multi-Branch PINN Training Results',
         ha='center', va='top', fontsize=17, fontweight='bold', color=TEXT)
fig.text(0.5, 0.932,
         'Best epoch 23  ·  Val loss 0.1149  ·  Wildfire recall 95.4%  ·  False-alarm AUC 1.0000  ·  Val accuracy 95.8%  ·  2023 fully held-out year',
         ha='center', va='top', fontsize=10, color=SUBTEXT)

# ── Stat callouts ─────────────────────────────────────────────────────────────
stats_text = [
    ('95.4%', 'Wildfire recall',     ORANGE, 0.10),
    ('99.9%', 'False-alarm recall',  NAVY,   0.30),
    ('1.0000','False-alarm AUC',     TEAL,   0.50),
    ('95.8%', 'Val accuracy (2023)', GOLD,   0.70),
    ('202K',  'Model parameters',    PURPLE, 0.88),
]
for val_txt, lbl_txt, col, xpos in stats_text:
    fig.text(xpos, 0.905, val_txt, ha='center', va='top',
             fontsize=15, fontweight='bold', color=col)
    fig.text(xpos, 0.882, lbl_txt, ha='center', va='top',
             fontsize=8, color=SUBTEXT)

# Divider line below callouts
fig.add_artist(plt.Line2D([0.06, 0.97], [0.875, 0.875], color=BORDER,
                           linewidth=0.8, transform=fig.transFigure))

# ── Footer ────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.015,
         'FireSight-IR  ·  github.com/Ibekwemmanuel7  ·  Physics-informed wildfire detection  ·  VIIRS + ERA5 + MODIS  ·  FireSat Protoflight-aligned',
         ha='center', va='bottom', fontsize=8, color=SUBTEXT)

out = f'{FIGURES_DIR}/03a_linkedin_hero.png'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'\n✓ LinkedIn hero figure saved → {out}')

---
## Section 3 — Standalone probability separation figure

This figure shows the key scientific result: the model's wildfire probability scores
perfectly separate false-alarms from wildfires — explaining why AUC = 1.0000.

In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG)
fig2.patch.set_facecolor(BG)

# Panel 1: wildfire probability by true class
ax = axes[0]; ax.set_facecolor(PANEL)
for ci, (name, color) in enumerate(zip(CLS_NAMES, CLS_COLORS)):
    sub = val_probs[val_labels==ci, 1]
    if len(sub) < 10: continue
    xr  = np.linspace(0, 1, 500)
    kde = stats.gaussian_kde(sub, bw_method=0.07)
    d   = kde(xr); d = d/d.max()
    alpha = 0.3 if ci == 1 else 0.5
    lw    = 3 if ci == 2 else 2.5
    ax.fill_between(xr, d, alpha=alpha, color=color)
    ax.plot(xr, d, color=color, lw=lw, label=f'{name}\n(n={len(sub):,}  med={np.median(sub):.3f})')
    # Median line
    ax.axvline(np.median(sub), color=color, lw=1.2, linestyle='--', alpha=0.6)

ax.axvline(0.5, color=GOLD, lw=2, linestyle=':', alpha=0.8)
ax.text(0.52, 0.97, 'Decision\nboundary', va='top', fontsize=9, color=GOLD)

# Annotation for perfect separation
ax.annotate('False-alarm scores\ncluster near 0\n→ AUC = 1.0000',
            xy=(0.05, 0.85), xytext=(0.15, 0.55),
            fontsize=9, color=NAVY,
            arrowprops=dict(arrowstyle='->', color=NAVY, lw=1.5))
ax.annotate('Wildfire scores\ncluster near 1',
            xy=(0.92, 0.78), xytext=(0.65, 0.55),
            fontsize=9, color=ORANGE,
            arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.5))

ax.set_xlabel('P(wildfire)  —  Model output score', fontsize=11, color=SUBTEXT)
ax.set_ylabel('Normalised density', fontsize=11, color=SUBTEXT)
ax.set_title('Model probability separation\n(2023 validation year — never seen during training)',
             color=TEXT, fontsize=12, pad=10)
ax.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=9, loc='center')
ax.spines[['top','right']].set_visible(False)
ax.grid(alpha=0.25)

# Panel 2: BTD vs BT_I4 scatter coloured by prediction
ax2 = axes[1]; ax2.set_facecolor(PANEL)
# Subsample for clarity
rng2 = np.random.default_rng(123)
for ci, (name, color) in enumerate(zip(CLS_NAMES, CLS_COLORS)):
    mask = val_preds == ci
    n_s  = min(5000, mask.sum())
    if n_s < 10: continue
    idx  = rng2.choice(np.where(mask)[0], n_s, replace=False)
    s_size = 6 if ci == 0 else 4
    alpha  = 0.25 if ci == 1 else 0.60
    ax2.scatter(val_btd[idx], val_bt4[idx], s=s_size, c=color,
                alpha=alpha, linewidths=0,
                label=f'{name} (pred)  n={n_s:,}')

ax2.axhline(310, color=GOLD, lw=1.5, linestyle=':', alpha=0.7, label='BT_I4=310K (physics min)')
ax2.axvline(10,  color=RED,  lw=1.5, linestyle=':', alpha=0.7, label='BTD=10K (physics min)')
ax2.set_xlabel('BTD = BT_I4 − BT_I5  (K)', fontsize=11, color=SUBTEXT)
ax2.set_ylabel('BT_I4 centre pixel  (K)', fontsize=11, color=SUBTEXT)
ax2.set_xlim(-5, 80); ax2.set_ylim(270, 500)
ax2.set_title('Physics constraint space\n(predicted class vs sensor observations)',
              color=TEXT, fontsize=12, pad=10)
ax2.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8,
           markerscale=3, loc='upper left')
ax2.spines[['top','right']].set_visible(False)
ax2.grid(alpha=0.2)

# Annotate physics quadrants
ax2.text(1, 308, 'Physics\nviolation\nzone', fontsize=7.5, color=RED, alpha=0.8, va='top')
ax2.text(55, 490, 'Active\nfire region', fontsize=8, color=ORANGE, alpha=0.9, va='top', ha='center')
ax2.text(2, 490, 'False-alarm\nregion', fontsize=8, color=NAVY, alpha=0.9, va='top')

fig2.text(0.5, 0.99,
          'FireSight-IR  |  Module 3a — What the Model Learned',
          ha='center', va='top', fontsize=15, fontweight='bold', color=TEXT)
fig2.text(0.5, 0.95,
          'Physics-informed loss produces perfectly calibrated probability scores  ·  False-alarm AUC = 1.0000  ·  Epoch 23  ·  Val 2023',
          ha='center', va='top', fontsize=10, color=SUBTEXT)
fig2.text(0.5, 0.01,
          'FireSight-IR  ·  github.com/Ibekwemmanuel7  ·  VIIRS + ERA5 + MODIS  ·  FireSat Protoflight-aligned',
          ha='center', va='bottom', fontsize=8, color=SUBTEXT)

plt.tight_layout(rect=[0, 0.02, 1, 0.93])
out2 = f'{FIGURES_DIR}/03a_probability_separation.png'
plt.savefig(out2, dpi=200, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'✓ Saved → {out2}')

---
## Section 4 — Figure inventory

In [ ]:
print('=== Module 3a LinkedIn Figures ===')
figs = [
    ('03a_linkedin_hero.png',         'BEST for LinkedIn — 4-panel composite'),
    ('03a_probability_separation.png','Physics interpretation — prob + BTD scatter'),
    ('03a_training_curves.png',       'Training curves (from 03a notebook, if exists)'),
    ('03a_confusion_matrix.png',      'Confusion matrices (from 03a notebook, if exists)'),
    ('03a_roc_curves.png',            'ROC curves (from 03a notebook, if exists)'),
]
for fname, desc in figs:
    fpath = f'{FIGURES_DIR}/{fname}'
    e = '✓' if os.path.exists(fpath) else '○'
    sz = f'{os.path.getsize(fpath)/1e6:.1f} MB' if os.path.exists(fpath) else 'not generated'
    print(f'  {e}  {fname:<42} {sz:>8}   {desc}')

print(f'\n  Recommended for LinkedIn: 03a_linkedin_hero.png')
print(f'  Share that image here and I will write the LinkedIn post.')